Идея проекта
Проект представляет собой веб-приложение и API для автоматической обработки аудиозаписей, которое объединяет три последовательных этапа анализа:

*   Транскрипция — преобразование речи в текст с помощью модели распознавания речи (ASR).
*   Суммаризация — автоматическое создание краткого резюме полученного текста.
*   Извлечение именованных сущностей (NER) — распознавание в тексте упоминаний людей, организаций, локаций, дат и других сущностей.

На вход пользователь подаёт аудиофайл (запись совещания, лекции, интервью, звонка), на выходе получает:

1.   полную расшифровку текста;
2.   краткое резюме (3–5 предложений);
3.   структурированный список ключевых сущностей (кто, где, что упоминалось).

Возможные области применения:
1. Бизнес и корпоративная среда. Автоматическое протоколирование совещаний: расшифровка + краткое резюме + список упомянутых участников и компаний
2. Медиа и журналистика. Быстрая обработка интервью: расшифровка записи, выделение ключевых персон и организаций для последующей публикации
3. Образование. Обработка записей лекций и вебинаров: студент получает текст, краткий конспект и список упомянутых терминов/имён
4. Колл-центры и поддержка. Анализ записей звонков: расшифровка диалога, автоматическое определение темы обращения и извлечение упомянутых данных (имена, адреса, компании)
5. Юридическая сфера. Обработка записей судебных заседаний и допросов: расшифровка + выделение ключевых лиц, организаций и мест
6. Медицина. Расшифровка диктовок врачей: текст приёма + краткое резюме + извлечение упомянутых лекарств, диагнозов, ФИО пациента
7. Подкасты и YouTube. Автоматическая генерация субтитров, описаний и тегов для публикации контента
8. Исследования. Обработка аудиоархивов: фольклорные записи, диалектологические интервью, социологические опросы

Ключевая ценность проекта — экономия времени. Часовая запись совещания, на прослушивание и конспектирование которой уходит 2–3 часа, обрабатывается автоматически за несколько минут, а пользователь сразу получает структурированный результат.

  Проект построен на трёх предварительно обученных моделях с платформы Hugging Face. Обучение с нуля не требуется — все модели готовы к использованию «из коробки».

Модель транскрипции: Whisper
* Название: openai/whisper-small

* Ссылка: https://huggingface.co/openai/whisper-small

* Задача: Automatic Speech Recognition (ASR) — преобразование речи в текст.

* Разработчик: OpenAI

* Описание: Whisper — мультиязычная модель, обученная на 680 000 часов аудиоданных. Поддерживает распознавание речи на 99 языках, устойчива к шуму, акцентам и фоновым звукам. Вариант small (244M параметров) выбран как компромисс между качеством и скоростью работы на CPU.


Модель суммаризации: BART
* Название: facebook/bart-large-cnn

* Ссылка: https://huggingface.co/facebook/bart-large-cnn

* Задача: Abstractive Summarization (абстрактная суммаризация) — генерация краткого пересказа текста своими словами.

* Разработчик: Meta AI (Facebook AI Research)

* Описание: BART — модель encoder-decoder на основе трансформеров, дообученная на датасете CNN/DailyMail для задач суммаризации новостных статей. Хорошо работает на текстах средней и большой длины, генерирует связные и осмысленные резюме.

Модель NER: русскоязычная
* Основная модель: Gherman/bert-base-NER-Russian

* Ссылка: https://huggingface.co/Gherman/bert-base-NER-Russian

* Описание: Дообученная версия bert-base-multilingual-cased для распознавания именованных сущностей в русском тексте. Распознаёт: имена, фамилии, отчества, страны, регионы, города, районы, улицы, дома. F1 Score — 0.988.


Для проекта «Транскрипция аудио + саммари + NER» используются три предварительно обученные модели с Hugging Face.

1.1. Модели и их назначение
Транскрипция	openai/whisper-small или faster-whisper	Преобразование аудио в текст
Суммаризация	facebook/bart-large-cnn	Создание краткого резюме из текста
NER	dslim/bert-base-NER	Извлечение именованных сущностей (люди, организации, локации)

Установка зависимостей:

In [ ]:
!pip install -q transformers torch datasets sentencepiece protobuf
!pip install -q faster-whisper  # для быстрой транскрипции

Загрузка и тестирование модели Whisper (транскрипция):

In [8]:
from transformers import pipeline
import torch

transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1
)

# Готовый русскоязычный пример из открытого датасета
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"

result = transcriber(audio_url, generate_kwargs={"language": "english"})
print("📝 Транскрипция:", result["text"])

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

📝 Транскрипция:  He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick, peppered, flour-fattened sauce.


Тестирование модели BART (суммаризация):

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Русскоязычная модель суммаризации
model_name = "cointegrated/rut5-base-absum"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Русскоязычный пример текста
russian_text = """
В деревне, затерянной среди зелёных холмов, жил старик по имени Иван.
Каждое утро он выходил на крыльцо своего дома и смотрел на восход солнца.
Иван любил свою деревню и её жителей. Однажды в деревню приехал молодой
врач по имени Анна. Она открыла небольшую клинику и начала лечить местных
жителей. Иван подружился с Анной, и они вместе организовали сбор средств
на строительство новой школы для детей деревни.
"""

# Токенизация и генерация
inputs = tokenizer(
    russian_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=80,
        min_length=20,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("📋 Резюме:", summary)

config.json:   0%|          | 0.00/753 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  828kB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  977MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


📋 Резюме: В деревне, затерянной среди зелёных холмов, жил старик по имени Иван. Он любил свою деревню и её жителей.


Тестирование модели BERT для NER (извлечение сущностей):

In [4]:
from transformers import pipeline

# Загрузка русскоязычной NER-модели
ner_pipeline = pipeline(
    "token-classification",
    model="Gherman/bert-base-NER-Russian",
    aggregation_strategy="simple"
)

# Русскоязычный пример текста
russian_text = "Меня зовут Иван Иванович из Москвы. Я работаю в компании Сбербанк."

# Извлечение сущностей
entities = ner_pipeline(russian_text)

for entity in entities:
    print(f"🔹 {entity['word']} → {entity['entity_group']} (уверенность: {entity['score']:.2f})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🔹 Иван → FIRST_NAME (уверенность: 1.00)
🔹 Иванович → MIDDLE_NAME (уверенность: 1.00)
🔹 Москвы → CITY (уверенность: 1.00)
